In [3]:
import os
import sys
import pandas as pd
import numpy as np
import glob
import time
import scipy
from scipy.sparse import csr_matrix
import anndata as an
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import random
from importlib import reload
import warnings
import ot
from scipy.spatial.distance import pdist, squareform
from matplotlib.colors import ListedColormap
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from kneed import KneeLocator  

"""WARNING: no warnings"""
warnings.filterwarnings("ignore")

source_path = os.path.abspath("../utilities/calculations/")
sys.path.append(source_path)
import centrality as central

In [4]:
%%time 
# 1MB resolution
resolution = 1000000

fpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/anndata/population_mESC_{resolution}_features.h5ad"

adata_main = sc.read_h5ad(fpath)

sc.logging.print_memory_usage()



Memory usage: current 3.20 GB, difference +3.20 GB
CPU times: user 12.5 s, sys: 2.57 s, total: 15.1 s
Wall time: 18.1 s


# General Data Scrubbing and adding the Centrality and Curvature Scores

In [5]:
# Carbon Copying the main adata so I do not have to pull the data set every single time.
adata = adata_main

In [6]:
def find_outliers_iqr(df_column):
  """
  Identifies outliers in a pandas DataFrame column using the IQR method.

  Args:
    df_column: A pandas Series representing the column to analyze.

  Returns:
    A boolean mask with True for outliers and False otherwise.
  """
  Q1 = df_column.quantile(0.25)
  Q3 = df_column.quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  return (df_column < lower_bound) | (df_column > upper_bound)

adata.obs['degree_outlier'] = find_outliers_iqr(adata.obs['degree'])

adata.obs[adata.obs['degree_outlier']][['chrom_bin', 'degree', 'degree_outlier']].head()

,chrom_bin,degree,degree_outlier
bin_name,,,
chr8:21,21,12931,True
chr13:120,120,2329,True
chrX:139,139,1609,True
chrX:122,122,1726,True
chrX:123,123,610,True


In [7]:
print(adata.obs['degree_outlier'].sum())
print(adata.obs['degree_outlier'].dtype)
adata.obs.shape

244
bool


(2579, 23)

In [8]:
mask = adata.obs['degree_outlier'].fillna(False).astype(bool).values
print(f"Removing {mask.sum()} outlier loci")
print(adata.obs_names[mask].tolist())
adata = adata[~mask, :].copy()

print(adata.obs.shape)

print('done!')

Removing 244 outlier loci
['chr8:21', 'chr13:120', 'chrX:139', 'chrX:122', 'chrX:123', 'chr4:156', 'chr8:20', 'chrX:124', 'chr17:30', 'chr1:85', 'chrX:110', 'chrX:52', 'chrX:14', 'chrX:63', 'chrX:169', 'chr14:18', 'chrX:168', 'chr8:71', 'chrX:9', 'chrX:165', 'chr6:47', 'chrX:154', 'chr9:124', 'chrX:15', 'chrX:82', 'chr9:3', 'chrX:6', 'chrX:12', 'chrX:61', 'chrX:88', 'chrX:76', 'chrX:109', 'chrX:29', 'chrX:27', 'chrX:137', 'chr14:19', 'chrX:102', 'chrX:84', 'chrX:120', 'chrX:97', 'chrX:41', 'chrX:24', 'chr6:90', 'chrX:56', 'chrX:28', 'chrX:118', 'chr9:88', 'chr8:19', 'chrX:156', 'chr1:24', 'chr8:11', 'chrX:108', 'chrX:135', 'chrX:68', 'chr15:99', 'chr14:42', 'chrX:42', 'chr14:14', 'chr4:3', 'chrX:70', 'chrX:11', 'chrX:111', 'chr14:15', 'chr13:119', 'chr8:106', 'chrX:130', 'chr14:16', 'chrX:80', 'chrX:64', 'chr17:40', 'chrX:134', 'chr18:90', 'chrX:127', 'chrX:155', 'chr15:74', 'chrX:74', 'chr8:70', 'chrX:18', 'chrX:141', 'chrX:45', 'chrX:121', 'chr4:112', 'chrX:59', 'chr11:3', 'chr12:3',

In [9]:
fpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_hge_logexp_nodes.csv"
bpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_poset_curvature_nodes.csv"
df_1 = pd.read_csv(fpath)
df_2 = pd.read_csv(bpath)

print(f"{df_1.shape=}")
print(df_1.columns)
print(f"{df_2.shape=}")
print(df_2.columns)

df_1.shape=(2335, 5)
Index(['bin_name', 'chrom', 'bin_start', 'bin_end',
       'global_hge_logexp_unweighted'],
      dtype='object')
df_2.shape=(2335, 10)
Index(['node_id', 'bin_name', 'chrom_bin', 'chrom', 'bin_start', 'bin_end',
       'degree', 'scalar_curvature', 'normalized_scalar_curvature',
       'abs_normalized_edge_curvature'],
      dtype='object')


In [10]:
#columns_to_drop = [x for x in df.columns if x in adata.obs.columns]
columns_to_drop = ["chrom", "bin_start", "bin_end"]
df_1 = df_1.drop(columns=columns_to_drop)
print(f"{df_1.shape=}")
df_1.head()

df_1.shape=(2335, 2)


,bin_name,global_hge_logexp_unweighted
0,chr9:121,0.000428
1,chr19:26,0.000421
2,chr4:127,0.000433
3,chr10:57,0.000425
4,chr12:8,0.000426


In [11]:
columns_to_drop_2 = ["chrom", "bin_start", "bin_end", "node_id", "chrom_bin", "degree"]
df_2 = df_2.drop(columns=columns_to_drop_2)
print(f"{df_2.shape=}")
df_2.head()

df_2.shape=(2335, 4)


,bin_name,scalar_curvature,normalized_scalar_curvature,abs_normalized_edge_curvature
0,chr9:121,-9300499,-3045.350033,3045.350033
1,chr19:26,-8314541,-2880.991337,2880.991337
2,chr4:127,-10683981,-3262.284275,3262.284275
3,chr10:57,-7936616,-2815.401206,2815.401206
4,chr12:8,-9343189,-3056.326137,3056.326137


In [12]:
adata.obs = pd.merge(
    adata.obs, df_1.set_index('bin_name'),
    how='left',
    left_index=True,
    right_index=True,
)

adata.obs = pd.merge(
    adata.obs, df_2.set_index('bin_name'),
    how='left',
    left_index=True,
    right_index=True,
)

In [13]:
cpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_poset_curvature_edges.csv"
dpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_hge_logexp_edges.csv"
df_3 = pd.read_csv(cpath)
df_4 = pd.read_csv(dpath)
    
columns_to_drop_3 = ["read_index", "read_length_bp", "degree", "order"]
columns_to_drop_4 = ["read_index", "n_bins"]

df_3 = df_3.drop(columns=columns_to_drop_3)
df_4 = df_4.drop(columns=columns_to_drop_4)
print(f"{df_3.shape=}")
print(f"{df_4.shape=}")
df_3.head()
df_4.head()

df_3.shape=(2039865, 4)
df_4.shape=(2039865, 2)


,read_name,global_hge_logexp_unweighted
0,3891ee6d-53d1-4ee0-ba2f-3d22291d4493,1.346761e-06
1,66953ddf-e76d-4cdf-aaf8-be028a2d7b04,3.508383e-30
2,ad5b2240-893f-4ed0-a157-c2be66d8d754,1.994061e-23
3,207b28dd-a1ae-443a-9c8e-d09a2dd018dd,1.337412e-06
4,3f354c45-5e48-4f6d-8c7e-05369432b344,1.293423e-06


In [14]:
adata.var = pd.merge(
    adata.var, df_3.set_index('read_name'),
    how='left',
    left_index=True,
    right_index=True,
)
adata.var = pd.merge(
    adata.var, df_4.set_index('read_name'),
    how='left',
    left_index=True,
    right_index=True,
)

In [15]:
cols = [
    "global_hge_logexp_unweighted",
    "edge_curvature",
    "normalized_edge_curvature",
    "abs_normalized_edge_curvature",
]

# Keep only vars where ALL these columns are non-NaN
keep_mask = adata.var[cols].notna().all(axis=1)

print(f"Removing {(~keep_mask).sum()} vars with NaNs")

adata = adata[:, keep_mask.values].copy()


Removing 716602 vars with NaNs


In [16]:
# filter
H = adata.X.tocsr().astype(float)
print(f"raw shape = {H.shape}")

raw shape = (2335, 2039865)


In [17]:
# drop singletons (reads with < 2 loci after row filter)
col_sums = np.asarray(H.sum(axis=0)).ravel()
col_mask = col_sums >= 2
H = H[:, col_mask]
var_idx = np.where(col_mask)[0]          # maps H cols -> adata.var cols

In [18]:
# drop large hyperedges (reads with > 10 loci)
col_sums = np.asarray(H.sum(axis=0)).ravel()
col_mask = col_sums <= 10
H = H[:, col_mask]
var_idx = var_idx[col_mask]

print(f"filtered shape = {H.shape}")

filtered shape = (2335, 2039865)


In [19]:
print(f"filtered shape = {H.shape}")
print(f"df_1 rows: {len(df_1)}  (should equal H rows: {H.shape[0]})")
print(f"df_2 rows: {len(df_2)}  (should equal H rows: {H.shape[0]})")
assert len(df_1) == H.shape[0], "Mismatch: check that df_1 was built from the same obs_idx filter"
assert len(df_2) == H.shape[0], "Mismatch: check that df_2 was built from the same obs_idx filter"

filtered shape = (2335, 2039865)
df_1 rows: 2335  (should equal H rows: 2335)
df_2 rows: 2335  (should equal H rows: 2335)


### Defining Kneedle Threshold

In [20]:
cntrl_scores = df_1['global_hge_logexp_unweighted']
curve_scores = df_2['abs_normalized_edge_curvature']

In [21]:
cntrl_sorted_scores = np.sort(cntrl_scores)[::-1]
z_scores_cntrl = (cntrl_scores - cntrl_scores.mean()) / cntrl_scores.std()

print("\nThreshold options & nodes selected:")
thresholds = {
    "Top 1%  (percentile)":    np.percentile(cntrl_scores, 99),
    "Top 5%  (percentile)":    np.percentile(cntrl_scores, 95),
    "Top 10% (percentile)":    np.percentile(cntrl_scores, 90),
    "Top 20% (percentile)":    np.percentile(cntrl_scores, 80),
    "Z-score > 2":             cntrl_scores.mean() + 2 *  cntrl_scores.std(),
    "Z-score > 1.5":           cntrl_scores.mean() + 1.5 * cntrl_scores.std(),
    "IQR outliers (Q3+1.5IQR)":np.percentile(cntrl_scores, 75) + 1.5 * (np.percentile(cntrl_scores, 75) - np.percentile(cntrl_scores, 25)),
}


for label, thresh in thresholds.items():
    n = (cntrl_scores >= thresh).sum()
    print(f"  {label:<35s}  cutoff={thresh:.7f}  → {n:7d} nodes  ({100*n/len(cntrl_scores):.1f}%)")

# Kneedle elbow on sorted scores
try:
    kl = KneeLocator(np.arange(len(cntrl_sorted_scores)), cntrl_sorted_scores,
                     curve="convex", direction="decreasing", interp_method="polynomial")
    knee_rank  = kl.knee
    cntrl_knee_score = cntrl_sorted_scores[knee_rank] if knee_rank is not None else None
    n_knee = (cntrl_scores >= cntrl_knee_score).sum() if cntrl_knee_score is not None else None
    print(f"  {'Kneedle elbow':<35s}  cutoff={cntrl_knee_score:.7f}  → {n_knee:7d} nodes  ({100*n_knee/len(cntrl_scores):.1f}%)")
    thresholds["Kneedle elbow"] = cntrl_knee_score
except Exception as e:
    print(f"  Kneedle failed: {e}")

THRESHOLD_LABEL =  "Kneedle elbow"
THRESHOLD_CENTRALITY = thresholds[THRESHOLD_LABEL]


Threshold options & nodes selected:
  Top 1%  (percentile)                 cutoff=0.0004410  →      24 nodes  (1.0%)
  Top 5%  (percentile)                 cutoff=0.0004362  →     117 nodes  (5.0%)
  Top 10% (percentile)                 cutoff=0.0004341  →     234 nodes  (10.0%)
  Top 20% (percentile)                 cutoff=0.0004320  →     467 nodes  (20.0%)
  Z-score > 2                          cutoff=0.0004383  →      68 nodes  (2.9%)
  Z-score > 1.5                        cutoff=0.0004358  →     145 nodes  (6.2%)
  IQR outliers (Q3+1.5IQR)             cutoff=0.0004404  →      30 nodes  (1.3%)
  Kneedle elbow                        cutoff=0.0004337  →     257 nodes  (11.0%)


In [22]:
curve_sorted_scores = np.sort(curve_scores)[::-1]

curve_med = df_2['abs_normalized_edge_curvature'].median()
print(curve_med)

lower = []
upper = []
for k in df_2['abs_normalized_edge_curvature']:
    if k > curve_med:
        upper.append(k)
    else: 
        lower.append(k)
curve_lower = np.array(lower)
curve_upper = np.array(upper)

crv_lwr_srt = np.sort(curve_lower)[::-1]
crv_upr_srt = np.sort(curve_upper)[::-1]

z_scores_curve = (curve_scores - curve_scores.mean()) / curve_scores.std()

print("\nThreshold options & nodes selected:")
thresholds = {
    "Top 1%  (percentile)":    np.percentile(curve_scores, 99),
    "Top 5%  (percentile)":    np.percentile(curve_scores, 95),
    "Top 10% (percentile)":    np.percentile(curve_scores, 90),
    "Top 20% (percentile)":    np.percentile(curve_scores, 80),
    "Z-score > 2":             curve_scores.mean() + 2 *  curve_scores.std(),
    "Z-score > 1.5":           curve_scores.mean() + 1.5 * curve_scores.std(),
    "IQR outliers (Q3+1.5IQR)":np.percentile(curve_scores, 75) + 1.5 * (np.percentile(curve_scores, 75) - np.percentile(curve_scores, 25)),
}


for label, thresh in thresholds.items():
    n = (curve_scores >= thresh).sum()
    print(f"  {label:<35s}  cutoff={thresh:.7f}  → {n:7d} nodes  ({100*n/len(curve_scores):.1f}%)")

# Kneedle elbow on sorted scores
try:
    kl = KneeLocator(np.arange(len(crv_lwr_srt)), crv_lwr_srt,
                     curve="convex", direction="decreasing", interp_method="polynomial")
    knee_rank  = kl.knee
    curve_lwr_knee = crv_lwr_srt[knee_rank] if knee_rank is not None else None
    n_knee = (curve_lower >= curve_lwr_knee).sum() if curve_lwr_knee is not None else None
    print(f"  {'Kneedle elbow - Lower':<35s}  cutoff={curve_lwr_knee:.7f}  → {n_knee:7d} nodes  ({100*n_knee/len(curve_lower):.1f}%)")
    thresholds["Kneedle elbow - Lower"] = curve_lwr_knee
except Exception as e:
    print(f"  Kneedle failed: {e}")

try:
    kl = KneeLocator(np.arange(len(crv_upr_srt)), crv_upr_srt,
                     curve="convex", direction="decreasing", interp_method="polynomial")
    knee_rank  = kl.knee
    curve_upr_knee = crv_upr_srt[knee_rank] if knee_rank is not None else None
    n_knee = (curve_upper >= curve_upr_knee).sum() if curve_upr_knee is not None else None
    print(f"  {'Kneedle elbow - Upper':<35s}  cutoff={curve_upr_knee:.7f}  → {n_knee:7d} nodes  ({100*n_knee/len(curve_upper):.1f}%)")
    thresholds["Kneedle elbow - Upper"] = curve_upr_knee
except Exception as e:
    print(f"  Kneedle failed: {e}")


THRESHOLD_LABEL =  "Kneedle elbow - Lower"
THRESHOLD_CURVATURE_LOWER = thresholds[THRESHOLD_LABEL]

THRESHOLD_LABEL =  "Kneedle elbow - Upper"
THRESHOLD_CURVATURE_UPPER = thresholds[THRESHOLD_LABEL]

3016.2335428382403

Threshold options & nodes selected:
  Top 1%  (percentile)                 cutoff=3686.9435842  →      24 nodes  (1.0%)
  Top 5%  (percentile)                 cutoff=3452.1980553  →     117 nodes  (5.0%)
  Top 10% (percentile)                 cutoff=3330.6206015  →     234 nodes  (10.0%)
  Top 20% (percentile)                 cutoff=3204.7838383  →     467 nodes  (20.0%)
  Z-score > 2                          cutoff=3571.0409819  →      61 nodes  (2.6%)
  Z-score > 1.5                        cutoff=3423.6597057  →     140 nodes  (6.0%)
  IQR outliers (Q3+1.5IQR)             cutoff=3728.9814714  →       6 nodes  (0.3%)
  Kneedle elbow - Lower                cutoff=3009.9697775  →      19 nodes  (1.6%)
  Kneedle elbow - Upper                cutoff=3296.8241525  →     288 nodes  (24.7%)


## Core Nodes Creation

In [23]:
df_total = pd.merge(
    df_1.set_index('bin_name'), df_2.set_index('bin_name'),
    how='left',
    left_index=True,
    right_index=True,
)
df_total.head()

,global_hge_logexp_unweighted,scalar_curvature,normalized_scalar_curvature,abs_normalized_edge_curvature
bin_name,,,,
chr9:121,0.000428,-9300499,-3045.350033,3045.350033
chr19:26,0.000421,-8314541,-2880.991337,2880.991337
chr4:127,0.000433,-10683981,-3262.284275,3262.284275
chr10:57,0.000425,-7936616,-2815.401206,2815.401206
chr12:8,0.000426,-9343189,-3056.326137,3056.326137


In [24]:
mask_interior = (df_total['global_hge_logexp_unweighted'] > THRESHOLD_CENTRALITY) & (df_total['abs_normalized_edge_curvature'] > THRESHOLD_CURVATURE_UPPER)
node_interior= df_total.index[mask_interior].tolist()

mask_bridge = (df_total['global_hge_logexp_unweighted'] > THRESHOLD_CENTRALITY) & (df_total['abs_normalized_edge_curvature'] < THRESHOLD_CURVATURE_LOWER)
node_bridge = df_total.index[mask_bridge].tolist()

node_core = node_interior + node_bridge

In [25]:
core_mask = adata.obs_names.isin(node_core)

node_core_df = adata.obs[core_mask].copy()
node_periph_df = adata.obs[~core_mask].copy()

node_core_df = node_core_df.rename(columns={'scalar_curvature': 'node_curvature',
                                            'normalized_scalar_curvature': 'normalized_node_curvature',
                                            'abs_normalized_edge_curvature': 'abs_normalized_node_curvature'})
node_periph_df = node_periph_df.rename(columns={'scalar_curvature': 'node_curvature',
                                            'normalized_scalar_curvature': 'normalized_node_curvature',
                                            'abs_normalized_edge_curvature': 'abs_normalized_node_curvature'})

node_core_df.head()

,bin_index,bin_start,bin_end,bin,chrom,chrom_bin,degree,genes,n_genes,ATACSeq_1,...,RNA_3,RNA_4,RNA_5,RNA_6,PolII,degree_outlier,global_hge_logexp_unweighted,node_curvature,normalized_node_curvature,abs_normalized_node_curvature
bin_name,,,,,,,,,,,,,,,,,,,,,
chr3:34,16,34000000,35000000,412,3,34,4126,Gm9791;Sox2ot;Dnajc19;Gm34599;Gm21388;Gm29135;...,23,0.654700,...,0.187619,0.425621,1.903160,0.135107,0.848217,False,0.000437,-12680004,-3553.812780,3553.812780
chr6:93,21,93000000,94000000,940,6,93,4187,Gm5313;Gm22312;Gm44220;9530026P05Rik;Magi1;Gm4...,10,0.706136,...,0.140705,0.090659,0.162810,0.099323,0.487627,False,0.000437,-13225816,-3632.468003,3632.468003
chr3:32,42,32000000,33000000,410,3,32,4048,Kcnmb3;Mrpl47;Gm37834;Usp13;Zfp639;Gm37770;Kcn...,16,0.672969,...,0.194078,0.193396,0.705671,0.154875,1.295672,False,0.000437,-12603137,-3548.180462,3548.180462
chr18:3,46,3000000,4000000,2322,18,3,4248,Gm50073;Gm50075;Cul2;Gm50088;G430049J08Rik;Cre...,20,0.466249,...,0.154656,0.151137,0.219633,0.113674,0.416700,False,0.000440,-13224779,-3634.179445,3634.179445
chr12:113,55,113000000,114000000,1764,12,113,4244,Ighd5-3;Ighd5-1;Ighv2-5;Ighv5-12;Ighj4;Ighv5-4...,94,0.668532,...,0.187602,0.182494,0.592320,0.158550,0.721855,False,0.000439,-13737222,-3702.755256,3702.755256


## Defining Kneedle Threshold - Edges

In [26]:
cntrl_scores_edges = df_4['global_hge_logexp_unweighted']
curve_scores_edges = df_3['abs_normalized_edge_curvature']

In [27]:
cntrl_scores = cntrl_scores_edges

cntrl_sorted_scores = np.sort(cntrl_scores)[::-1]
z_scores_cntrl = (cntrl_scores - cntrl_scores.mean()) / cntrl_scores.std()

print("\nThreshold options & nodes selected:")
thresholds = {
    "Top 1%  (percentile)":    np.percentile(cntrl_scores, 99),
    "Top 5%  (percentile)":    np.percentile(cntrl_scores, 95),
    "Top 10% (percentile)":    np.percentile(cntrl_scores, 90),
    "Top 20% (percentile)":    np.percentile(cntrl_scores, 80),
    "Z-score > 2":             cntrl_scores.mean() + 2 *  cntrl_scores.std(),
    "Z-score > 1.5":           cntrl_scores.mean() + 1.5 * cntrl_scores.std(),
    "IQR outliers (Q3+1.5IQR)":np.percentile(cntrl_scores, 75) + 1.5 * (np.percentile(cntrl_scores, 75) - np.percentile(cntrl_scores, 25)),
}


for label, thresh in thresholds.items():
    n = (cntrl_scores >= thresh).sum()
    print(f"  {label:<35s}  cutoff={thresh:.7f}  → {n:7d} nodes  ({100*n/len(cntrl_scores):.1f}%)")

# Kneedle elbow on sorted scores
try:
    kl = KneeLocator(np.arange(len(cntrl_sorted_scores)), cntrl_sorted_scores,
                     curve="convex", direction="decreasing", interp_method="polynomial")
    knee_rank  = kl.knee
    cntrl_knee_score = cntrl_sorted_scores[knee_rank] if knee_rank is not None else None
    n_knee = (cntrl_scores >= cntrl_knee_score).sum() if cntrl_knee_score is not None else None
    print(f"  {'Kneedle elbow':<35s}  cutoff={cntrl_knee_score:.7f}  → {n_knee:7d} nodes  ({100*n_knee/len(cntrl_scores):.1f}%)")
    thresholds["Kneedle elbow"] = cntrl_knee_score
except Exception as e:
    print(f"  Kneedle failed: {e}")

THRESHOLD_LABEL =  "Kneedle elbow"
THRESHOLD_CENTRALITY = thresholds[THRESHOLD_LABEL]


Threshold options & nodes selected:
  Top 1%  (percentile)                 cutoff=0.0000014  →   20399 nodes  (1.0%)
  Top 5%  (percentile)                 cutoff=0.0000014  →  102024 nodes  (5.0%)
  Top 10% (percentile)                 cutoff=0.0000014  →  203987 nodes  (10.0%)
  Top 20% (percentile)                 cutoff=0.0000013  →  407973 nodes  (20.0%)
  Z-score > 2                          cutoff=0.0000018  →       0 nodes  (0.0%)
  Z-score > 1.5                        cutoff=0.0000015  →       0 nodes  (0.0%)
  IQR outliers (Q3+1.5IQR)             cutoff=0.0000033  →       0 nodes  (0.0%)
  Kneedle elbow                        cutoff=0.0000014  →  102927 nodes  (5.0%)


In [28]:
curve_scores = curve_scores_edges
curve_sorted_scores = np.sort(curve_scores)[::-1]

curve_med = df_3['abs_normalized_edge_curvature'].median()
print(curve_med)

lower = []
upper = []
for k in df_3['abs_normalized_edge_curvature']:
    if k > curve_med:
        upper.append(k)
    else: 
        lower.append(k)
curve_lower = np.array(lower)
curve_upper = np.array(upper)

crv_lwr_srt = np.sort(curve_lower)[::-1]
crv_upr_srt = np.sort(curve_upper)[::-1]

z_scores_curve = (curve_scores - curve_scores.mean()) / curve_scores.std()

print("\nThreshold options & nodes selected:")
thresholds = {
    "Top 1%  (percentile)":    np.percentile(curve_scores, 99),
    "Top 5%  (percentile)":    np.percentile(curve_scores, 95),
    "Top 10% (percentile)":    np.percentile(curve_scores, 90),
    "Top 20% (percentile)":    np.percentile(curve_scores, 80),
    "Z-score > 2":             curve_scores.mean() + 2 *  curve_scores.std(),
    "Z-score > 1.5":           curve_scores.mean() + 1.5 * curve_scores.std(),
    "IQR outliers (Q3+1.5IQR)":np.percentile(curve_scores, 75) + 1.5 * (np.percentile(curve_scores, 75) - np.percentile(curve_scores, 25)),
}


for label, thresh in thresholds.items():
    n = (curve_scores >= thresh).sum()
    print(f"  {label:<35s}  cutoff={thresh:.7f}  → {n:7d} nodes  ({100*n/len(curve_scores):.1f}%)")

# Kneedle elbow on sorted scores
try:
    kl = KneeLocator(np.arange(len(crv_lwr_srt)), crv_lwr_srt,
                     curve="convex", direction="decreasing", interp_method="polynomial")
    knee_rank  = kl.knee
    curve_lwr_knee = crv_lwr_srt[knee_rank] if knee_rank is not None else None
    n_knee = (curve_lower >= curve_lwr_knee).sum() if curve_lwr_knee is not None else None
    print(f"  {'Kneedle elbow - Lower':<35s}  cutoff={curve_lwr_knee:.7f}  → {n_knee:7d} nodes  ({100*n_knee/len(curve_lower):.1f}%)")
    thresholds["Kneedle elbow - Lower"] = curve_lwr_knee
except Exception as e:
    print(f"  Kneedle failed: {e}")

try:
    kl = KneeLocator(np.arange(len(crv_upr_srt)), crv_upr_srt,
                     curve="convex", direction="decreasing", interp_method="polynomial")
    knee_rank  = kl.knee
    curve_upr_knee = crv_upr_srt[knee_rank] if knee_rank is not None else None
    n_knee = (curve_upper >= curve_upr_knee).sum() if curve_upr_knee is not None else None
    print(f"  {'Kneedle elbow - Upper':<35s}  cutoff={curve_upr_knee:.7f}  → {n_knee:7d} nodes  ({100*n_knee/len(curve_upper):.1f}%)")
    thresholds["Kneedle elbow - Upper"] = curve_upr_knee
except Exception as e:
    print(f"  Kneedle failed: {e}")


THRESHOLD_LABEL =  "Kneedle elbow - Lower"
THRESHOLD_CURVATURE_LOWER = thresholds[THRESHOLD_LABEL]

THRESHOLD_LABEL =  "Kneedle elbow - Upper"
THRESHOLD_CURVATURE_UPPER = thresholds[THRESHOLD_LABEL]

1157.2

Threshold options & nodes selected:
  Top 1%  (percentile)                 cutoff=3216.2500000  →   20401 nodes  (1.0%)
  Top 5%  (percentile)                 cutoff=3025.6666667  →  102085 nodes  (5.0%)
  Top 10% (percentile)                 cutoff=2793.6666667  →  204030 nodes  (10.0%)
  Top 20% (percentile)                 cutoff=2205.8571429  →  407979 nodes  (20.0%)
  Z-score > 2                          cutoff=3119.9489658  →   53117 nodes  (2.6%)
  Z-score > 1.5                        cutoff=2672.3461475  →  225709 nodes  (11.1%)
  IQR outliers (Q3+1.5IQR)             cutoff=4326.9835681  →       0 nodes  (0.0%)
  Kneedle elbow - Lower                cutoff=516.6458333  →  450151 nodes  (44.1%)
  Kneedle elbow - Upper                cutoff=2228.0000000  →  396193 nodes  (38.8%)


## Core Edges Creation

In [29]:
RHO = 0.65
KAPPA = 0.35

# --- Setup ---
# H: (n_nodes × n_edges) sparse incidence matrix
# core_nodes: list/array of node names that are in V_core
# adata.var has columns 'centrality' and 'curvature' (placeholders — rename to match yours)

# 1. Build a boolean vector marking core nodes (length n_nodes)
core_mask = adata.obs_names.isin(node_core)            # shape (n_nodes,)
core_vec = core_mask.astype(float).reshape(1, -1)        # shape (1, n_nodes)

# 2. |E ∩ V_core| for each edge = sum over core-node rows of H, per column
#    Equivalent to: core_vec @ H  -> shape (1, n_edges)
core_count_per_edge = np.asarray(core_vec @ H).ravel()   # shape (n_edges,)

# 3. |E| for each edge = total nonzeros per column
#    If H is weighted and you want size = count of members, use (H != 0).sum(axis=0).
#    If membership is just 0/1, H.sum(axis=0) works too.
edge_size = np.asarray((H != 0).sum(axis=0)).ravel()     # shape (n_edges,)

# 4. Fraction of each edge that lies in the core
core_fraction = np.divide(
    core_count_per_edge, edge_size,
    out=np.zeros_like(core_count_per_edge, dtype=float),
    where=edge_size > 0,
)

# 5. Pull centrality and curvature out of adata.var
x_E = adata.var['global_hge_logexp_unweighted'].values
ric = adata.var['abs_normalized_edge_curvature'].values

# 6. Build the three condition masks
high_centrality = x_E > THRESHOLD_CENTRALITY
high_curvature  = ric > THRESHOLD_CURVATURE_UPPER
low_curvature   = ric < THRESHOLD_CURVATURE_LOWER
in_core_enough_interior = core_fraction > RHO   # RHO = 0.5 to start
in_core_enough_bridge = core_fraction > KAPPA   # KAPPA = 0.5 to start

# 7. Partition
E_int_mask    = high_centrality & high_curvature & in_core_enough_interior
E_bridge_mask = high_centrality & low_curvature  & in_core_enough_bridge
E_peri_mask   = ~(E_int_mask | E_bridge_mask)




# 8. Store the partition label on adata.var so it's easy to filter later
labels = np.full(adata.n_vars, 'periphery', dtype=object)
labels[E_int_mask]    = 'interior'
labels[E_bridge_mask] = 'bridge'
adata.var['edge_class'] = labels

# Convenience: get the edge IDs in each set
E_int   = adata.var_names[E_int_mask].tolist()
E_bridge = adata.var_names[E_bridge_mask].tolist()
E_peri   = adata.var_names[E_peri_mask].tolist()

edge_core = E_int + E_bridge

In [30]:
core_mask = adata.var_names.isin(edge_core)

edge_core_df = adata.var[core_mask].copy()
edge_periph_df = adata.var[~core_mask].copy()

In [31]:
edge_core_df.head()

,read_index,basename,mean_mapq,median_mapq,n_chromosomes,order,n_bins,read_length_bp,genes,n_genes,edge_curvature,normalized_edge_curvature,abs_normalized_edge_curvature,global_hge_logexp_unweighted,edge_class
read_name,,,,,,,,,,,,,,,
906345b0-7338-4a82-8af1-2b7e1fbebb28,9,batch03,24.750000,19.0,1,4,3,5261,Myo16;Shcbp1,2,-10181.0,-242.404762,242.404762,0.000001,bridge
f35dd078-bf11-441c-a2c1-a830f395ff91,32,batch04,40.333333,60.0,1,3,3,4715,Csmd1,1,-94578.0,-359.612167,359.612167,0.000001,bridge
22753480-398c-4771-9d88-6db9fa09f860,311,batch03,50.444444,60.0,1,9,3,4783,Gprc5a;Dusp16;Fam234b;Hebp1;Zxdc,5,-34083.0,-248.781022,248.781022,0.000001,bridge
a0035c9a-f816-46ae-8de5-fc1cd9cd611f,316,batch03,60.000000,60.0,1,3,2,2978,,0,-10941.0,-280.538462,280.538462,0.000001,bridge
8dfc201e-d9cc-4490-9a64-b7b352dbefa5,355,batch01,52.428571,60.0,1,7,2,4972,Ermard;Fam120b;Gm51425,3,-27308.0,-227.566667,227.566667,0.000001,bridge


In [32]:
node_core_df.index.name = 'bin_name'

node_core_df.head()


,bin_index,bin_start,bin_end,bin,chrom,chrom_bin,degree,genes,n_genes,ATACSeq_1,...,RNA_3,RNA_4,RNA_5,RNA_6,PolII,degree_outlier,global_hge_logexp_unweighted,node_curvature,normalized_node_curvature,abs_normalized_node_curvature
bin_name,,,,,,,,,,,,,,,,,,,,,
chr3:34,16,34000000,35000000,412,3,34,4126,Gm9791;Sox2ot;Dnajc19;Gm34599;Gm21388;Gm29135;...,23,0.654700,...,0.187619,0.425621,1.903160,0.135107,0.848217,False,0.000437,-12680004,-3553.812780,3553.812780
chr6:93,21,93000000,94000000,940,6,93,4187,Gm5313;Gm22312;Gm44220;9530026P05Rik;Magi1;Gm4...,10,0.706136,...,0.140705,0.090659,0.162810,0.099323,0.487627,False,0.000437,-13225816,-3632.468003,3632.468003
chr3:32,42,32000000,33000000,410,3,32,4048,Kcnmb3;Mrpl47;Gm37834;Usp13;Zfp639;Gm37770;Kcn...,16,0.672969,...,0.194078,0.193396,0.705671,0.154875,1.295672,False,0.000437,-12603137,-3548.180462,3548.180462
chr18:3,46,3000000,4000000,2322,18,3,4248,Gm50073;Gm50075;Cul2;Gm50088;G430049J08Rik;Cre...,20,0.466249,...,0.154656,0.151137,0.219633,0.113674,0.416700,False,0.000440,-13224779,-3634.179445,3634.179445
chr12:113,55,113000000,114000000,1764,12,113,4244,Ighd5-3;Ighd5-1;Ighv2-5;Ighv5-12;Ighj4;Ighv5-4...,94,0.668532,...,0.187602,0.182494,0.592320,0.158550,0.721855,False,0.000439,-13737222,-3702.755256,3702.755256


## Creating a Sub Matrix of H using Core

In [33]:
node_mask = adata.obs_names.isin(node_core)
edge_mask = adata.var_names.isin(edge_core)

adata_core = adata[node_mask, edge_mask].copy()
H_core = adata_core.X.tocsr().astype(float)

In [34]:
adata_core

AnnData object with n_obs × n_vars = 195 × 29588
    obs: 'bin_index', 'bin_start', 'bin_end', 'bin', 'chrom', 'chrom_bin', 'degree', 'genes', 'n_genes', 'ATACSeq_1', 'ATACSeq_2', 'ATACSeq_3', 'CTCF', 'H3K27ac', 'H3K27me3', 'RNA_1', 'RNA_2', 'RNA_3', 'RNA_4', 'RNA_5', 'RNA_6', 'PolII', 'degree_outlier', 'global_hge_logexp_unweighted', 'scalar_curvature', 'normalized_scalar_curvature', 'abs_normalized_edge_curvature'
    var: 'read_index', 'basename', 'mean_mapq', 'median_mapq', 'n_chromosomes', 'order', 'n_bins', 'read_length_bp', 'genes', 'n_genes', 'edge_curvature', 'normalized_edge_curvature', 'abs_normalized_edge_curvature', 'global_hge_logexp_unweighted', 'edge_class'
    uns: 'base_resolution', 'chrom_sizes', 'gdf', 'gene_map', 'intervals'
    layers: 'H'

## Write Cores to CSV

In [35]:
node_core_df.index.name = 'bin_name'
node_periph_df.index.name = 'bin_name'
node_core_df.to_csv("/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_node_core.csv", index=True)
node_periph_df.to_csv("/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_node_periph.csv", index=True)

node_core_df.index.name = 'read_name'
node_periph_df.index.name = 'read_name'
edge_core_df.to_csv("/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_edge_core.csv", index=True)
edge_periph_df.to_csv("/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_edge_periph.csv", index=True)


In [36]:
sp.save_npz("/scratch/indikar_root/indikar1/jduhamel/pore_c/H_core.npz", H_core.tocsr())

In [37]:
zdata = an.AnnData(X=H_core, obs=node_core_df, var=edge_core_df)

zdata.shape

zdata.write('/scratch/indikar_root/indikar1/jduhamel/pore_c/population_mESC_100000_core.h5ad')